# Corte 2 · Modelo, consulta y limpieza de datos — Spotify
**Ciencia de Datos · Actividad calificable · CORHUILA 2026-B**

Este cuaderno (1) carga el CSV de Kaggle [3], (2) mide su calidad **antes** de limpiar, (3) aplica la limpieza paso a paso, (4) mide **después**, (5) carga el resultado en las tablas del ERD y (6) responde dos preguntas con consultas (pandas y SQL). Las cinco dimensiones de calidad usadas (completitud, exactitud, consistencia, unicidad y vigencia) son las del OVA de la Semana 9 [1].

> Cuaderno elaborado con asistencia de inteligencia artificial (Claude, Anthropic).

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

RAW = Path("data/spotify_raw.csv")
CLEAN = Path("data/spotify_limpio.csv")
pd.set_option("display.max_columns", None, "display.width", 140)

## 1. Carga y diagnóstico ANTES de limpiar

In [2]:
raw = pd.read_csv(RAW)

def perfil(df, etiqueta):
    cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
    espacios = sum(int((df[c].astype(str) != df[c].astype(str).str.strip()).sum()) for c in cat_cols)
    return {
        "etapa": etiqueta,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos_totales": int(df.isna().sum().sum()),
        "duplicados_exactos": int(df.duplicated().sum()),
        "memoria_KB": round(df.memory_usage(deep=True).sum() / 1024, 1),
        "celdas_con_espacios": espacios,
    }

antes = perfil(raw, "ANTES")
print(antes)
print()
print(raw.dtypes.to_string())
print("\nNulos por columna:\n", raw.isna().sum().to_string())

{'etapa': 'ANTES', 'filas': 500, 'columnas': 12, 'nulos_totales': 0, 'duplicados_exactos': 0, 'memoria_KB': np.float64(170.5), 'celdas_con_espacios': 0}

Country                                str
Artist                                 str
Album                                  str
Genre                                  str
Release Year                         int64
Monthly Listeners (Millions)       float64
Total Streams (Millions)           float64
Total Hours Streamed (Millions)    float64
Avg Stream Duration (Min)          float64
Platform Type                          str
Streams Last 30 Days (Millions)    float64
Skip Rate (%)                      float64

Nulos por columna:
 Country                            0
Artist                             0
Album                              0
Genre                              0
Release Year                       0
Monthly Listeners (Millions)       0
Total Streams (Millions)           0
Total Hours Streamed (Millions)    0
Avg Stream Du

**Diagnóstico adicional (exactitud, unicidad y coherencia).** Los nulos y duplicados exactos no son todo: se revisan reglas de negocio.

In [3]:
CLAVE = ["Country", "Album", "Genre", "Platform Type"]   # segmento: país x álbum x género x plan

dup_clave = int(raw.duplicated(CLAVE).sum())
viol_30d = int((raw["Streams Last 30 Days (Millions)"] > raw["Total Streams (Millions)"]).sum())
fuera_skip = int((~raw["Skip Rate (%)"].between(0, 100)).sum())
dur_invalida = int((raw["Avg Stream Duration (Min)"] <= 0).sum())
ratio = raw["Total Hours Streamed (Millions)"] / (raw["Total Streams (Millions)"] * raw["Avg Stream Duration (Min)"])

print("Filas repetidas por segmento (misma clave, distintas métricas):", dup_clave)
print("Filas con streams de 30 días > streams totales (imposible):  ", viol_30d)
print("Skip rate fuera de 0-100:", fuera_skip, "| duración <= 0:", dur_invalida)
print("Horas / (streams x duración en min): mediana =", round(ratio.median(), 3),
      "(si 'horas' fueran horas reales, debería ser ~0.017 = 1/60)")

Filas repetidas por segmento (misma clave, distintas métricas): 26
Filas con streams de 30 días > streams totales (imposible):   2
Skip rate fuera de 0-100: 0 | duración <= 0: 0
Horas / (streams x duración en min): mediana = 0.998 (si 'horas' fueran horas reales, debería ser ~0.017 = 1/60)


Hallazgos del diagnóstico: no hay nulos ni duplicados exactos, pero sí (a) **filas repetidas por segmento**, (b) **filas imposibles** (los streams de 30 días no pueden superar los totales) y (c) una **inconsistencia de unidad**: la columna de horas se comporta como *minutos* (≈ streams × duración), no como horas.

## 2. Limpieza paso a paso

In [4]:
bitacora = []   # registro de cada transformación (documentación)
df = raw.copy()

# 2.1 Normalizar nombres de columna a snake_case
RENOMBRAR = {
    "Country": "pais", "Artist": "artista", "Album": "album", "Genre": "genero",
    "Release Year": "anio_lanzamiento", "Monthly Listeners (Millions)": "oyentes_mensuales_m",
    "Total Streams (Millions)": "streams_totales_m", "Total Hours Streamed (Millions)": "horas_reportadas_m",
    "Avg Stream Duration (Min)": "duracion_prom_min", "Platform Type": "plan",
    "Streams Last 30 Days (Millions)": "streams_30d_m", "Skip Rate (%)": "skip_rate_pct",
}
df = df.rename(columns=RENOMBRAR)
bitacora.append(("Nombres", f"{len(RENOMBRAR)} columnas renombradas a snake_case"))

# 2.2 Nulos: imputar mediana (numéricas) o 'Desconocido' (texto). Aquí hay 0, pero la regla queda aplicada.
num_cols = df.select_dtypes("number").columns
txt_cols = ["pais", "artista", "album", "genero", "plan"]
n_nulos = int(df.isna().sum().sum())
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[txt_cols] = df[txt_cols].fillna("Desconocido")
bitacora.append(("Nulos", f"{n_nulos} encontrados -> imputación por mediana/'Desconocido' (0 celdas modificadas)"))

# 2.3 Formato de texto: quitar espacios y unificar mayúsculas en catálogos (no en nombres propios)
cambios_fmt = 0
for col in txt_cols:
    limpio = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    cambios_fmt += int((limpio != df[col]).sum())
    df[col] = limpio
df["plan"] = df["plan"].str.title()
bitacora.append(("Formato", f"{cambios_fmt} celdas con espacios corregidas; plan en formato Title"))

# 2.4 Tipos: texto repetitivo -> category; año -> entero pequeño; métricas -> float
df[txt_cols] = df[txt_cols].astype("category")
df["anio_lanzamiento"] = pd.to_numeric(df["anio_lanzamiento"], errors="raise").astype("int16")
df[list(num_cols.drop("anio_lanzamiento"))] = df[list(num_cols.drop("anio_lanzamiento"))].apply(pd.to_numeric, errors="raise")
bitacora.append(("Tipos", "5 columnas de texto -> category; anio_lanzamiento int64 -> int16"))

# 2.5 Duplicados exactos
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates()
bitacora.append(("Duplicados exactos", f"{n_dup} eliminados"))

# 2.6 Reglas de exactitud: eliminar filas imposibles
mask_ok = (df["streams_30d_m"] <= df["streams_totales_m"]) & df["skip_rate_pct"].between(0, 100) & (df["duracion_prom_min"] > 0)
n_inval = int((~mask_ok).sum())
df = df[mask_ok]
bitacora.append(("Exactitud", f"{n_inval} filas imposibles eliminadas (streams 30 días > totales)"))

# 2.7 Unicidad por segmento: una fila por país x álbum x género x plan (promedio de las observaciones repetidas)
CLAVE_L = ["pais", "album", "genero", "plan"]
n_rep = int(df.duplicated(CLAVE_L).sum())
agg = {c: "mean" for c in df.select_dtypes("number").columns if c != "anio_lanzamiento"}
agg.update({"artista": "first", "anio_lanzamiento": "first"})
df = (df.groupby(CLAVE_L, observed=True, as_index=False).agg(agg)
        .round(2).reset_index(drop=True))
df[txt_cols] = df[txt_cols].astype("category")
bitacora.append(("Unicidad", f"{n_rep} filas repetidas por segmento consolidadas (promedio de métricas)"))

# 2.8 Unidad coherente: horas reales esperadas = streams x duración / 60
df["horas_esperadas_m"] = (df["streams_totales_m"] * df["duracion_prom_min"] / 60).round(2)
bitacora.append(("Unidad", "columna 'horas_reportadas_m' conservada solo como referencia; se agrega 'horas_esperadas_m' = streams x min / 60"))

pd.DataFrame(bitacora, columns=["paso", "acción y resultado"])

,paso,acción y resultado
0,Nombres,12 columnas renombradas a snake_case
1,Nulos,0 encontrados -> imputación por mediana/'Desco...
2,Formato,0 celdas con espacios corregidas; plan en form...
3,Tipos,5 columnas de texto -> category; anio_lanzamie...
4,Duplicados exactos,0 eliminados
5,Exactitud,2 filas imposibles eliminadas (streams 30 días...
6,Unicidad,26 filas repetidas por segmento consolidadas (...
7,Unidad,columna 'horas_reportadas_m' conservada solo c...


## 3. Diagnóstico DESPUÉS y comparación

In [5]:
despues = perfil(df, "DESPUÉS")
comparacion = pd.DataFrame([antes, despues]).set_index("etapa").T
comparacion

etapa,ANTES,DESPUÉS
filas,500.0,472.0
columnas,12.0,13.0
nulos_totales,0.0,0.0
duplicados_exactos,0.0,0.0
memoria_KB,170.5,32.8
celdas_con_espacios,0.0,0.0


In [6]:
tipos = pd.DataFrame({"antes": raw.dtypes.astype(str).values}, index=list(raw.columns)).rename(index=RENOMBRAR)
tipos["después"] = df.dtypes.astype(str).reindex(tipos.index)
tipos

,antes,después
pais,str,category
artista,str,category
album,str,category
genero,str,category
anio_lanzamiento,int64,int16
oyentes_mensuales_m,float64,float64
streams_totales_m,float64,float64
horas_reportadas_m,float64,float64
duracion_prom_min,float64,float64
plan,str,category


In [7]:
CLEAN.parent.mkdir(exist_ok=True)
df.to_csv(CLEAN, index=False, encoding="utf-8")
print("Guardado:", CLEAN, "->", len(df), "filas")

Guardado: data/spotify_limpio.csv -> 472 filas


## 4. Carga en las tablas del ERD
El resultado limpio se separa en las seis entidades del ERD (`diagrama-erd-corte2.svg`) y se carga en SQLite con PK, FK y restricciones. Si alguna FK se violara, la carga fallaría.

In [8]:
def dim(serie, id_col):
    t = pd.DataFrame({"nombre": sorted(serie.astype(str).unique())})
    t.insert(0, id_col, range(1, len(t) + 1))
    return t

pais, genero, plan = dim(df["pais"], "country_id"), dim(df["genero"], "genero_id"), dim(df["plan"], "plan_id")
artista = dim(df["artista"], "artist_id")

album = (df[["artista", "album", "anio_lanzamiento"]].astype({"artista": str, "album": str}).drop_duplicates()
         .merge(artista, left_on="artista", right_on="nombre")
         .rename(columns={"album": "titulo"}).sort_values(["artist_id", "titulo"]).reset_index(drop=True))
album["album_id"] = album.index + 1
album = album[["album_id", "artist_id", "titulo", "anio_lanzamiento"]]

hechos = (df.astype({"pais": str, "album": str, "genero": str, "plan": str, "artista": str})
            .merge(artista.rename(columns={"nombre": "artista"}), on="artista")
            .merge(album.rename(columns={"titulo": "album"}), on=["artist_id", "album", "anio_lanzamiento"])
            .merge(pais.rename(columns={"nombre": "pais"}), on="pais")
            .merge(genero.rename(columns={"nombre": "genero"}), on="genero")
            .merge(plan.rename(columns={"nombre": "plan"}), on="plan"))
metrica = hechos[["country_id", "album_id", "genero_id", "plan_id", "oyentes_mensuales_m", "streams_totales_m",
                  "streams_30d_m", "duracion_prom_min", "skip_rate_pct", "horas_esperadas_m"]].reset_index(drop=True)
metrica.insert(0, "metric_id", metrica.index + 1)
assert len(metrica) == len(df)

DDL = '''
PRAGMA foreign_keys = ON;
CREATE TABLE pais    (country_id INT PRIMARY KEY, nombre VARCHAR(80) NOT NULL UNIQUE);
CREATE TABLE genero  (genero_id  INT PRIMARY KEY, nombre VARCHAR(40) NOT NULL UNIQUE);
CREATE TABLE plan    (plan_id    INT PRIMARY KEY, nombre VARCHAR(30) NOT NULL UNIQUE);
CREATE TABLE artista (artist_id  INT PRIMARY KEY, nombre VARCHAR(100) NOT NULL UNIQUE);
CREATE TABLE album   (album_id INT PRIMARY KEY, artist_id INT NOT NULL REFERENCES artista(artist_id),
                      titulo VARCHAR(150) NOT NULL, anio_lanzamiento SMALLINT NOT NULL,
                      UNIQUE (artist_id, titulo, anio_lanzamiento));
CREATE TABLE metrica_streaming (
  metric_id INT PRIMARY KEY,
  country_id INT NOT NULL REFERENCES pais(country_id),
  album_id   INT NOT NULL REFERENCES album(album_id),
  genero_id  INT NOT NULL REFERENCES genero(genero_id),
  plan_id    INT NOT NULL REFERENCES plan(plan_id),
  oyentes_mensuales_m DECIMAL(10,2) NOT NULL CHECK (oyentes_mensuales_m >= 0),
  streams_totales_m   DECIMAL(10,2) NOT NULL CHECK (streams_totales_m >= 0),
  streams_30d_m       DECIMAL(10,2) NOT NULL CHECK (streams_30d_m >= 0 AND streams_30d_m <= streams_totales_m),
  duracion_prom_min   DECIMAL(6,2)  NOT NULL CHECK (duracion_prom_min > 0),
  skip_rate_pct       DECIMAL(5,2)  NOT NULL CHECK (skip_rate_pct BETWEEN 0 AND 100),
  horas_esperadas_m   DECIMAL(12,2) NOT NULL,
  UNIQUE (country_id, album_id, genero_id, plan_id)
);
'''
con = sqlite3.connect(":memory:")
con.executescript(DDL)
for nombre, t in [("pais", pais), ("genero", genero), ("plan", plan), ("artista", artista),
                  ("album", album), ("metrica_streaming", metrica)]:
    t.to_sql(nombre, con, if_exists="append", index=False)
print("Violaciones de FK:", con.execute("PRAGMA foreign_key_check").fetchall() or "ninguna")

Violaciones de FK: ninguna


## 5. Pregunta 1 — ¿Qué países concentran la demanda reciente de usuarios Premium?
**Filtro:** plan = Premium. **Agregación:** suma de streams de los últimos 30 días por país (demanda reciente → capacidad de servidores, el caso del curso).

In [9]:
# pandas
q1_pd = (df[df["plan"] == "Premium"]
         .groupby("pais", observed=True)["streams_30d_m"].sum().round(2)
         .sort_values(ascending=False).head(5).rename("streams_30d_m").reset_index())
q1_pd

,pais,streams_30d_m
0,Italy,1901.34
1,South Africa,1688.34
2,Mexico,1685.91
3,United Kingdom,1550.61
4,Brazil,1429.94


In [10]:
# SQL equivalente (con JOIN al ERD)
q1_sql = pd.read_sql_query('''
    SELECT p.nombre AS pais, ROUND(SUM(m.streams_30d_m), 2) AS streams_30d_m
    FROM metrica_streaming m
    JOIN pais p ON p.country_id = m.country_id
    JOIN plan pl ON pl.plan_id = m.plan_id
    WHERE pl.nombre = 'Premium'
    GROUP BY p.nombre
    ORDER BY streams_30d_m DESC
    LIMIT 5
''', con)
assert q1_sql["pais"].tolist() == q1_pd["pais"].tolist(), "pandas y SQL deben coincidir"
q1_sql

,pais,streams_30d_m
0,Italy,1901.34
1,South Africa,1688.34
2,Mexico,1685.91
3,United Kingdom,1550.61
4,Brazil,1429.94


In [11]:
total_prem = df[df["plan"] == "Premium"]["streams_30d_m"].sum()
top3 = q1_pd["streams_30d_m"].head(3).sum()
print(f"Top 3 = {top3:,.1f} M de {total_prem:,.1f} M streams Premium (30 días) -> {top3 / total_prem:.1%}")

Top 3 = 5,275.6 M de 24,142.6 M streams Premium (30 días) -> 21.9%


## 6. Pregunta 2 — ¿Qué géneros se saltan más los oyentes en lanzamientos recientes?
**Filtro:** álbumes lanzados desde 2022. **Agregación:** promedio de `skip_rate_pct` y conteo de segmentos por género (señal para la curaduría regional).

In [12]:
q2_pd = (df[df["anio_lanzamiento"] >= 2022]
         .groupby("genero", observed=True)
         .agg(skip_rate_prom=("skip_rate_pct", "mean"), segmentos=("skip_rate_pct", "size"))
         .round(2).sort_values("skip_rate_prom", ascending=False).reset_index())
q2_pd

,genero,skip_rate_prom,segmentos
0,R&B,23.72,11
1,Pop,23.51,18
2,Rock,22.18,25
3,Hip Hop,21.82,21
4,Jazz,21.65,18
5,EDM,20.94,19
6,Reggaeton,20.55,21
7,Classical,18.52,25
8,Indie,16.46,19
9,K-pop,15.62,15


In [13]:
q2_sql = pd.read_sql_query('''
    SELECT g.nombre AS genero,
           ROUND(AVG(m.skip_rate_pct), 2) AS skip_rate_prom,
           COUNT(*) AS segmentos
    FROM metrica_streaming m
    JOIN genero g ON g.genero_id = m.genero_id
    JOIN album a  ON a.album_id  = m.album_id
    WHERE a.anio_lanzamiento >= 2022
    GROUP BY g.nombre
    ORDER BY skip_rate_prom DESC
''', con)
assert q2_sql["genero"].tolist() == q2_pd["genero"].tolist()
q2_sql

,genero,skip_rate_prom,segmentos
0,R&B,23.72,11
1,Pop,23.51,18
2,Rock,22.18,25
3,Hip Hop,21.82,21
4,Jazz,21.65,18
5,EDM,20.94,19
6,Reggaeton,20.55,21
7,Classical,18.52,25
8,Indie,16.46,19
9,K-pop,15.62,15


In [14]:
print("Rango de skip rate entre géneros:", q2_pd["skip_rate_prom"].min(), "a", q2_pd["skip_rate_prom"].max(),
      "| promedio general:", round(df[df["anio_lanzamiento"] >= 2022]["skip_rate_pct"].mean(), 2))

Rango de skip rate entre géneros: 15.62 a 23.72 | promedio general: 20.45
